### Importing Library


In [3]:
from pathlib import Path
import os
import pandas as pd
import numpy as np
import time
import asyncio

from langchain.document_loaders import PyPDFDirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema import Document
from pinecone import Pinecone

In [4]:
from dotenv import load_dotenv
load_dotenv(override = True)

os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')

#### STEP 1: Loading of the PDF Data


In [5]:
#load the pdf document
folder_path = 'data'
pdf_loader = PyPDFDirectoryLoader(folder_path)
raw_document = pdf_loader.load()

In [6]:
raw_document 

[Document(metadata={'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2026-01-05T23:04:38+01:00', 'author': '', 'moddate': '2026-01-05T23:04:38+01:00', 'title': 'Microsoft Word - Document2', 'source': 'data\\City-guides.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1'}, page_content='A. New York City Guide - GrandStay ProperƟes \nDocument ID: GSH-EXT-NYC-001 \nLast Updated: March 20, 2024 \nApplicable: All NYC properƟes \nTOP ATTRACTIONS: \n1. Must-See Landmarks: \no Statue of Liberty & Ellis Island \n\uf0a7 Ferry from BaƩery Park: $24.50 adults \n\uf0a7 Advance Ɵckets required (book 3+ months ahead) \n\uf0a7 First ferry: 8:30 AM, last return: 5:00 PM \no Empire State Building \n\uf0a7 Hours: 8:00 AM - 2:00 AM daily \n\uf0a7 Tickets: $44 standard, $77 express \n\uf0a7 Best Ɵmes: Early morning or aŌer 10:00 PM \no Metropolitan Museum of Art \n\uf0a7 Hours: Sunday-Thursday 10:00 AM - 5:00 PM, Friday-Saturday 10:00 AM - \n9:00 PM \n\uf0a7 Suggested admission:

#### STEP 2: Cleaning and Chunking of the text


In [7]:
# Cleaning of text
import re

def clean_pdf_text(text: str) -> str:
    # substitute unicode text format with empty string
    text = re.sub(r"\uf0a7", " ", text)

    # Remove one or more white space character and replace with empty string
    text = re.sub(r"\s+", " ", text)

    # Remove page with some digits and substitute with empty string
    text = re.sub(r"Page \d+", " ", text)

    # Remove strange Unicode artifacts
    text = re.sub(r"[\uf0a7\u2022\u25cf\u25aa\u200b]", " ", text)

    # Remove page numbers
    text = re.sub(r"Page\s+\d+(\s+of\s+\d+)?", " ", text, flags=re.IGNORECASE)

    # Remove headers/footers like "Confidential"
    text = re.sub(r"Confidential", " ", text, flags=re.IGNORECASE)

    # Remove repeated underscores or dashes
    text = re.sub(r"[_]{2,}", " ", text)
    text = re.sub(r"[-]{3,}", " ", text)

    # Remove excessive dots
    text = re.sub(r"\.{2,}", ".", text)

    # Remove multiple spaces
    text = re.sub(r"[ \t]+", " ", text)

    # Remove excessive blank lines
    text = re.sub(r"\n\s*\n+", "\n\n", text)

    # Remove spaces before punctuation
    text = re.sub(r"\s+([.,!?;:])", r"\1", text)

    # Remove spaces inside brackets
    text = re.sub(r"\(\s+", "(", text)
    text = re.sub(r"\s+\)", ")", text)

    # Remove URLs 
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)

    # Remove email addresses 
    text = re.sub(r"\S+@\S+", " ", text)

    # Remove non-printable characters
    text = re.sub(r"[^\x20-\x7E\n]", " ", text)

    # Remove page borders
    text = re.sub(r"\|", " ", text)

    return text.strip()

In [8]:
clean_docs = [
    
    Document(
        page_content = clean_pdf_text(doc.page_content),
        metadata = doc.metadata 
    )
    for doc in raw_document
]



In [9]:
clean_docs

[Document(metadata={'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2026-01-05T23:04:38+01:00', 'author': '', 'moddate': '2026-01-05T23:04:38+01:00', 'title': 'Microsoft Word - Document2', 'source': 'data\\City-guides.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1'}, page_content='A. New York City Guide - GrandStay Proper es Document ID: GSH-EXT-NYC-001 Last Updated: March 20, 2024 Applicable: All NYC proper es TOP ATTRACTIONS: 1. Must-See Landmarks: o Statue of Liberty & Ellis Island Ferry from Ba ery Park: $24.50 adults Advance  ckets required (book 3+ months ahead) First ferry: 8:30 AM, last return: 5:00 PM o Empire State Building Hours: 8:00 AM - 2:00 AM daily Tickets: $44 standard, $77 express Best  mes: Early morning or a er 10:00 PM o Metropolitan Museum of Art Hours: Sunday-Thursday 10:00 AM - 5:00 PM, Friday-Saturday 10:00 AM - 9:00 PM Suggested admission: $30 adults Free for NYC residents & students 2. Neighborhood Highlights: o Times Square: 

In [10]:
# chunking the data
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 600,
    chunk_overlap = 100
)

chunks = text_splitter.split_documents(clean_docs)
print(f"Created {len(chunks)} chunks from the PDF data")

Created 47 chunks from the PDF data


In [11]:
chunks

[Document(metadata={'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2026-01-05T23:04:38+01:00', 'author': '', 'moddate': '2026-01-05T23:04:38+01:00', 'title': 'Microsoft Word - Document2', 'source': 'data\\City-guides.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1'}, page_content='A. New York City Guide - GrandStay Proper es Document ID: GSH-EXT-NYC-001 Last Updated: March 20, 2024 Applicable: All NYC proper es TOP ATTRACTIONS: 1. Must-See Landmarks: o Statue of Liberty & Ellis Island Ferry from Ba ery Park: $24.50 adults Advance  ckets required (book 3+ months ahead) First ferry: 8:30 AM, last return: 5:00 PM o Empire State Building Hours: 8:00 AM - 2:00 AM daily Tickets: $44 standard, $77 express Best  mes: Early morning or a er 10:00 PM o Metropolitan Museum of Art Hours: Sunday-Thursday 10:00 AM - 5:00 PM, Friday-Saturday 10:00 AM - 9:00 PM Suggested admission: $30'),
 Document(metadata={'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'cr

#### Step 3: Generate Embeddings and Store in Vector Pinecone Database


In [12]:
from langchain.embeddings import OpenAIEmbeddings
pc = Pinecone(api_key = os.getenv("PINECONE_API_KEY"))
embedding = OpenAIEmbeddings()

C:\Users\aghos\AppData\Local\Temp\ipykernel_23964\1762158588.py:3: LangChainDeprecationWarning: The class `OpenAIEmbeddings` was deprecated in LangChain 0.0.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import OpenAIEmbeddings``.
  embedding = OpenAIEmbeddings()


In [13]:
from langchain_pinecone import PineconeVectorStore
from pinecone import Pinecone, ServerlessSpec
import os
import time

pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))

index_name = "aurora"
correct_dimension = 1536
cloud = "aws"
region = "us-east-1"

def wait_for_index_ready(index_name):
    while not pc.describe_index(index_name).status["ready"]:
        print("Waiting for index to be ready...")
        time.sleep(2)

def create_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=correct_dimension,
        metric="cosine",
        spec=ServerlessSpec(
            cloud=cloud,
            region=region
        )
    )
    print(f"Created new index: {index_name}")
    wait_for_index_ready(index_name)

def add_documents_to_index(index_name, chunks, embedding):
    print(f"Adding {len(chunks)} documents/chunks to index...")

    docsearch = PineconeVectorStore.from_documents(
        documents=chunks,
        index_name=index_name,
        embedding=embedding
    )

    print(f"Successfully added {len(chunks)} documents/chunks to index.")
    return docsearch


existing_indexes = pc.list_indexes().names()

if index_name in existing_indexes:
    index_description = pc.describe_index(index_name)
    existing_dimension = index_description.dimension

    print(f"Index already exists: {index_name}")
    print(f"Existing index dimension: {existing_dimension}")

    if existing_dimension == correct_dimension:
        print("Index has the correct dimension.")

        index = pc.Index(index_name)
        stats = index.describe_index_stats()
        total_vector_count = stats.total_vector_count

        if total_vector_count == 0:
            print("Index exists but is empty.")
            docsearch = add_documents_to_index(index_name, chunks, embedding)
        else:
            print(f"Index already contains {total_vector_count} vectors.")
            print("Skipping document upload.")

            docsearch = PineconeVectorStore.from_existing_index(
                index_name=index_name,
                embedding=embedding
            )

    else:
        print("Index has the wrong dimension.")
        print(f"Deleting index with dimension {existing_dimension}...")

        pc.delete_index(index_name)
        time.sleep(15)

        create_index(index_name)
        docsearch = add_documents_to_index(index_name, chunks, embedding)

else:
    print(f"No index found with name: {index_name}")

    create_index(index_name)
    docsearch = add_documents_to_index(index_name, chunks, embedding)

print("Vector store is ready.")
print(f"Successfully added {len(chunks)} documents/chunks to index.")

c:\Users\aghos\OneDrive\Documents\GitHub\Intelligent_travel_Assistant_platform_for_Aurora_Hospitality\aurora_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Index already exists: aurora
Existing index dimension: 1536
Index has the correct dimension.
Index already contains 94 vectors.
Skipping document upload.
Vector store is ready.
Successfully added 47 documents/chunks to index.


In [14]:
docsearch.similarity_search("Amenities & services (wifi, breakfast, gym, pool, spa)", k=5)

[Document(id='69bc13ae-ffc7-4f74-95f7-8856380db74a', metadata={'author': '', 'creationdate': '2026-01-05T22:42:22+01:00', 'creator': 'PyPDF', 'moddate': '2026-01-05T22:42:22+01:00', 'page': 3.0, 'page_label': '4', 'producer': 'Microsoft: Print To PDF', 'source': 'data\\GSH-POL-001_Policy_manuals_v3.2_20240315.pdf', 'title': 'Microsoft Word - Document1', 'total_pages': 4.0}, page_content='in public areas o Prohibited in dining areas, pool areas,  tness centers AMENITIES PROVIDED:   Pet bed and bowls upon request   Welcome treat at check-in   Nearby walking maps   Pet-si ng services available ($30/hour)'),
 Document(id='c02f8934-04c7-4dc9-bfd3-d6e9e1922d0b', metadata={'author': '', 'creationdate': '2026-01-05T22:42:22+01:00', 'creator': 'PyPDF', 'moddate': '2026-01-05T22:42:22+01:00', 'page': 3.0, 'page_label': '4', 'producer': 'Microsoft: Print To PDF', 'source': 'data\\GSH-POL-001_Policy_manuals_v3.2_20240315.pdf', 'title': 'Microsoft Word - Document1', 'total_pages': 4.0}, page_conten

#### step 4: Retrieval Aspect

In [15]:
from langchain.prompts import PromptTemplate

policy_prompt = PromptTemplate(
    input_variables=["context",
                      "question",
                      "guest_type",
                      "loyalty",
                      "city"],
    template="""You are a POLICY INTERPRETATION AGENT for Aurora Hospitality. You have access to the following context information:

    ROLE
    - Interpret hotel policies strictly and accurately.
    - Use only the retrieved policy text
    - Never infer or soften policy language
    - Once you cannot find an information, 
    just refer the user to the customer support team for further assistance while you answer the question you are confident of.
    - Do not tell the user where you are pulling the information from.

    CONTEXT
    Guest Type: {guest_type}
    Loyalty Tier: {loyalty}
    City: {city}

    POLICY DOCUMENTS
    {context}

    USER QUESTION
    {question}

    OUTPUT FORMAT: A detailed information that respond clearly to the user query
   
    """
    #  OUTPUT FORMAT(JSON ONLY):
    # {{
    #     "policy_facts":[string],
    #     "limitations":[string],
    #     "applicability":string,
    #     "confidence": number
    # }}
    )
print(policy_prompt)

input_variables=['city', 'context', 'guest_type', 'loyalty', 'question'] input_types={} partial_variables={} template='You are a POLICY INTERPRETATION AGENT for Aurora Hospitality. You have access to the following context information:\n\n    ROLE\n    - Interpret hotel policies strictly and accurately.\n    - Use only the retrieved policy text\n    - Never infer or soften policy language\n    - Once you cannot find an information, \n    just refer the user to the customer support team for further assistance while you answer the question you are confident of.\n    - Do not tell the user where you are pulling the information from.\n\n    CONTEXT\n    Guest Type: {guest_type}\n    Loyalty Tier: {loyalty}\n    City: {city}\n\n    POLICY DOCUMENTS\n    {context}\n\n    USER QUESTION\n    {question}\n\n    OUTPUT FORMAT: A detailed information that respond clearly to the user query\n   \n    '


#### Step 5: Initialize LLM



In [16]:
from langchain.chains import ConversationalRetrievalChain
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(temperature=0, model_name="gpt-4o-mini")

In [17]:
policy_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=docsearch.as_retriever(),
    combine_docs_chain_kwargs={"prompt": policy_prompt},
    return_source_documents=False
)

In [18]:
chat_history = []
query = "Amenities & services (wifi, breakfast, gym, pool, spa)"

result = policy_chain.invoke(
    {"question": query, 
     "chat_history": chat_history, 
     "guest_type": "Business", 
     "loyalty": "Gold", 
     "city": "New York"})

chat_history.append((query, result["answer"]))
print(result["answer"])

For your stay, the following amenities and services are available:

1. **Wi-Fi**: The policy document does not specify details about Wi-Fi availability. Please contact customer support for more information.

2. **Breakfast**: The policy document does not provide information regarding breakfast services. Please reach out to customer support for clarification.

3. **Gym**: The policy document does not mention gym facilities. For details, please consult customer support.

4. **Pool**: The property has a pool, but specific details about its amenities or services are not provided. Please contact customer support for more information.

5. **Spa**: Spa services are available from 8:00 AM to 8:00 PM daily. Advance reservations are recommended, and a 24-hour cancellation notice is required.

For any additional inquiries or specific details, please reach out to customer support.


#### Step 6: Load Conversation Data

In [19]:
#Load conversation data
import json

conversation_file_path = 'data/gsh_support_chats_jan_jun_2024.json'

with open(conversation_file_path, 'r', encoding='utf-8') as f:
    conversation_data = json.load(f)

    print(f"{len(conversation_data)} Conversations loaded successfully.")

50 Conversations loaded successfully.


In [20]:
conversation_docs = []

for conversation in conversation_data:
    guest_message = [m['text'] for m in conversation['messages'] if m['sender'] == 'guest']
    assistant_message = [m['text'] for m in conversation['messages'] if m['sender'] == 'assistant']

    grounding_sources = []
    for msg in conversation['messages']:
        if msg.get('grounding_sources_type') and msg.get('grounding_sources_type') != 'none':
            grounding_sources.append(msg['grounding_sources_type'])
        unique_grounding_sources = list(set(grounding_sources))

    doc_content = f"""
        CONVERSATION ID: {conversation['conversation_id']},
        INTENT: {conversation['intent_primary']},
        SECONDARY INTENT: {conversation['intent_secondary'] if conversation['intent_secondary'] else 'None'},
        BRAND: {conversation['brand']},
        CITY: {conversation['property']['city']},
        COUNTRY: {conversation['property']['country']},
        GUEST TYPE: {conversation['guest_profile']['guest_type']},
        LOYALTY TIER: {conversation['guest_profile']['loyalty_tier']},
        RESOLUTION STATUS: {conversation['resolution']['status']},
        CSAT SCORE: {conversation['resolution']['csat_score']},
        TAGS: {', '.join(conversation['tags']) if conversation['tags'] else 'None'}

        KEY GUEST QUESTIONS: 
        {chr(10).join([f"- {msg}" for msg in guest_message[:3]])}
        
        SUCCESSFUL ASSISTANT RESPONSES:
        {chr(10).join([f"- {msg}" for msg in assistant_message[:3]])}

        GROUNDING SOURCES USED: {', '.join(unique_grounding_sources) if unique_grounding_sources else 'None'}

        CONVERSATION SUMMARY:
        This conversation shows how to handle {conversation['intent_primary']} inquiries for {conversation['guest_profile']['guest_type']} 
        with a {conversation['guest_profile']['loyalty_tier']} loyalty tier. The guest is from {conversation['property']['city']}, 
        {conversation['property']['country']}. The conversation was resolved with a status of {conversation['resolution']['status']} 
        and a CSAT score of {conversation['resolution']['csat_score']}.
        The assistant used sources: {', '.join(unique_grounding_sources[:2]) if unique_grounding_sources else 'general knowledge'} 
        to effectively address the guest's concerns, providing accurate information and guidance based on the hotel's policies and procedures.
 """     

    conversation_docs.append(Document(
        page_content=doc_content.strip(), 
        metadata={
            "type": "conversation",
            "conversation_id": conversation['conversation_id'],
            "intent_primary": conversation['intent_primary'],
            "secondary_intent": conversation['intent_secondary'] if conversation['intent_secondary'] else 'None',
            "brand": conversation['brand'],
            "city": conversation['property']['city'],
            "country": conversation['property']['country'],
            "guest_type": conversation['guest_profile']['guest_type'],
            "loyalty_tier": conversation['guest_profile']['loyalty_tier'],
            "resolution_status": conversation['resolution']['status'],
            "csat_score": conversation['resolution']['csat_score'],
            "tags": str(conversation['tags']),
            "source": "conversation history"
        }
            ))

    print(f"Created {len(conversation_docs)} conversation documents from the JSON data.")   

            

Created 1 conversation documents from the JSON data.
Created 2 conversation documents from the JSON data.
Created 3 conversation documents from the JSON data.
Created 4 conversation documents from the JSON data.
Created 5 conversation documents from the JSON data.
Created 6 conversation documents from the JSON data.
Created 7 conversation documents from the JSON data.
Created 8 conversation documents from the JSON data.
Created 9 conversation documents from the JSON data.
Created 10 conversation documents from the JSON data.
Created 11 conversation documents from the JSON data.
Created 12 conversation documents from the JSON data.
Created 13 conversation documents from the JSON data.
Created 14 conversation documents from the JSON data.
Created 15 conversation documents from the JSON data.
Created 16 conversation documents from the JSON data.
Created 17 conversation documents from the JSON data.
Created 18 conversation documents from the JSON data.
Created 19 conversation documents fro

#### step 7: Clean Conversation Docs

In [21]:
# Conversation vector store

conversation_index_name = "aurora-conversation-data"

cleaned_conversation_doc = []

for doc in conversation_docs:
    cleaned_metadata = {}
    for key, value in doc.metadata.items():
        if value is None:
            cleaned_metadata[key] = ""
        elif isinstance(value, list):
            cleaned_metadata[key] = str(value)
        elif isinstance(value, dict):
            cleaned_metadata[key] = str(value)
        else:
            cleaned_metadata[key] = value
    cleaned_doc = Document(
        page_content= doc.page_content,
        metadata = cleaned_metadata
    )
    cleaned_conversation_doc.append(cleaned_doc)

print(f"cleaned {len(cleaned_conversation_doc)}")




cleaned 50


#### Step 8: Store cleaned Conversation Doc in Pinecone vector Database

In [22]:
# storing conversational data to pinecone vector database
from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import PineconeVectorStore
import os
import time

pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))

conversation_index_name = "aurora-conversation-data"
correct_dimension = 1536
cloud = "aws"
region = "us-east-1"


def wait_for_index_ready(index_name):
    while not pc.describe_index(index_name).status["ready"]:
        print("Waiting for index to be ready...")
        time.sleep(2)


def create_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=correct_dimension,
        metric="cosine",
        spec=ServerlessSpec(
            cloud=cloud,
            region=region
        )
    )

    print(f"Created new index: {index_name}")
    wait_for_index_ready(index_name)


def add_documents_to_index(index_name, documents, embedding):
    print(f"Adding {len(documents)} documents to index...")

    vectorstore = PineconeVectorStore.from_documents(
        documents=cleaned_conversation_doc,
        index_name=index_name,
        embedding=embedding
    )

    print(f"Successfully added {len(documents)} documents to index.")
    return vectorstore


existing_indexes = pc.list_indexes().names()

if conversation_index_name in existing_indexes:
    index_description = pc.describe_index(conversation_index_name)
    existing_dimension = index_description.dimension

    print(f"Index already exists: {conversation_index_name}")
    print(f"Existing index dimension: {existing_dimension}")

    if existing_dimension == correct_dimension:
        print("Index has the correct dimension.")

        index = pc.Index(conversation_index_name)
        stats = index.describe_index_stats()
        total_vector_count = stats.total_vector_count

        if total_vector_count == 0:
            print("Index exists but is empty.")
            conversation_vectorstore = add_documents_to_index(
                conversation_index_name,
                cleaned_conversation_doc,
                embedding
            )
        else:
            print(f"Index already contains {total_vector_count} vectors.")
            print("Skipping document upload.")

            conversation_vectorstore = PineconeVectorStore.from_existing_index(
                index_name=conversation_index_name,
                embedding=embedding
            )

    else:
        print("Index has the wrong dimension.")
        print(f"Deleting index with dimension {existing_dimension}...")

        pc.delete_index(conversation_index_name)
        time.sleep(15)

        create_index(conversation_index_name)

        conversation_vectorstore = add_documents_to_index(
            conversation_index_name,
            cleaned_conversation_doc,
            embedding
        )

else:
    print(f"No index found with name: {conversation_index_name}")

    create_index(conversation_index_name)

    conversation_vectorstore = add_documents_to_index(
        conversation_index_name,
        cleaned_conversation_doc,
        embedding
    )

print("Vector store is ready.")

Index already exists: aurora-conversation-data
Existing index dimension: 1536
Index has the correct dimension.
Index already contains 50 vectors.
Skipping document upload.
Vector store is ready.


#### Step 9: Retieval Aspect for conversation agent

In [23]:
#Defining the prompt template for conversation agent

conversation_prompt = PromptTemplate(
    input_variables=["context", "question"],
    template="""You are a CONVERSATION AGENT for Aurora Hospitality, read through how agents handled similar 
    conversations in the past and answer your questions based on that information to provide accurate and 
    helpful responses to guests.

    ROLE
    - Analyze how similar conversations were handled in the past to provide accurate and helpful responses to guests.F
    - Focus on tone, escalation, uncertainty handling and resolution strategies used in previous conversations to guide your responses.
    - Once you cannot find an information, just refer the user to the customer support team for further 
    assistance while you answer the question you are confident of.
    - Do not tell the user where you are pulling the information from.
    - Do not invent policy

    HISTORICAL CONVERSATION
    {context}

    USER QUESTION
    {question}

     OUTPUT FORMAT(JSON ONLY):
    {{
        "observed_patterns":string,
        "response_style":string,
        "conversation":string,
        "confidence": number
    }}
    """
    )
print(conversation_prompt)

input_variables=['context', 'question'] input_types={} partial_variables={} template='You are a CONVERSATION AGENT for Aurora Hospitality, read through how agents handled similar \n    conversations in the past and answer your questions based on that information to provide accurate and \n    helpful responses to guests.\n\n    ROLE\n    - Analyze how similar conversations were handled in the past to provide accurate and helpful responses to guests.F\n    - Focus on tone, escalation, uncertainty handling and resolution strategies used in previous conversations to guide your responses.\n    - Once you cannot find an information, just refer the user to the customer support team for further \n    assistance while you answer the question you are confident of.\n    - Do not tell the user where you are pulling the information from.\n    - Do not invent policy\n\n    HISTORICAL CONVERSATION\n    {context}\n\n    USER QUESTION\n    {question}\n\n     OUTPUT FORMAT(JSON ONLY):\n    {{\n        "

In [24]:
from langchain.chains import RetrievalQA

conversation_retriever = conversation_vectorstore.as_retriever()

conversation_agent = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=conversation_retriever,
    chain_type_kwargs={"prompt": conversation_prompt},
    return_source_documents=False
)

In [25]:
chat_history = []
query = "Amenities & services (wifi, breakfast, gym, pool, spa)"

result = conversation_agent.invoke(
    {"query": query})
answer = result["result"]
chat_history.append((query, answer))
print(answer)

{
    "observed_patterns": "Guests frequently inquire about breakfast inclusion and Wi-Fi speed. The responses consistently clarify that breakfast inclusion depends on the rate and package, and that complimentary Wi-Fi is available throughout the hotel, supporting streaming and video calls. Breakfast hours are typically provided, with specific times for weekdays and weekends. The tone is friendly and informative, ensuring guests feel appreciated and well-informed.",
    "response_style": "The responses are concise, polite, and provide clear information. They often conclude with a friendly remark wishing the guest safe travels or asking if further assistance is needed.",
    "conversation": "Breakfast inclusion depends on your rate and package. Complimentary Wi-Fi is available throughout the hotel; streaming and video calls are supported on the standard network. Breakfast is typically served 6:30–10:30 AM on weekdays and 7:00–11:00 AM on weekends. If your property has seasonal adjustmen

#### Converting the policy agent and conversational agent to tools 
##### -Agents without Memory (Database)

In [26]:
# Policy interpretation agent tool
from langchain.tools import tool


@tool
def policy_agent_tool(question: str, guest_type: str, loyalty: str, city: str):
    """
    Tool for interpreting hotel policies based on guest information and location. It 
    retrieves relevant policy documents from the vector store and uses a language model 
    to provide accurate and detailed responses to user queries. The tool takes into account 
    the guest type, loyalty tier, and city to ensure that the information provided is applicable 
    to the specific context of the inquiry. The output reflects strict ploicies and procedures.
    """
    # Implementation for policy interpretation
    return policy_chain.invoke(
        {"question": question, 
         "chat_history": chat_history, 
         "guest_type": guest_type, 
         "loyalty": loyalty, 
         "city": city})["answer"]

In [27]:
# Conversational analyzer agent tool
@tool
def conversation_agent_tool(question: str):
    """
    Tool for analyzing historical conversations to provide insights and guidance on how to handle 
    similar inquiries. It retrieves relevant conversation data from the vector store and uses a 
    language model to analyze patterns, response styles, and resolution strategies. The tool 
    helps in understanding how similar situations were handled in the past, providing a basis for 
    informed decision-making in current interactions.
    """
    # Implementation for conversation analysis
    return conversation_agent.invoke(
        {"question": question})["result"]

#### Initialize Database 

In [ ]:
# Create a database that serves as agents memory
import sqlite3
db = sqlite3.connect('conversation_memory.db', check_same_thread=False)

db.execute(""" 
    CREATE TABLE IF NOT EXISTS conversation_memory(
        session_id TEXT,
        role TEXT,
        message TEXT,
        timestamp DATETIME DEFAULT CURRENT_TIMESTAMP
    )
""")

db.commit()

In [29]:
# Populate the database
def store_conversation(session_id: str, role: str, message: str):
    """
    Store a conversation message in the database with the session ID, role (user or assistant), 
    message content, and timestamp. This function allows for tracking and analyzing conversations 
    over time, providing a historical record of interactions for future reference and analysis.
    """
    db.execute(
        "INSERT INTO conversation_memory (session_id, role, message) VALUES (?, ?, ?)",
        (session_id, role, message)
    )
    db.commit()

In [30]:
# Getting each conversation one by one
def get_conversations(session_id: str, limit=6):
    """
    Retrieve a list of conversation messages from the database for a specific session ID, 
    limited to the most recent 'limit' number of messages. This function allows for fetching 
    historical conversation data for analysis or display, providing context for ongoing interactions.
    """
    rows = db.execute(
        """SELECT role, message, timestamp FROM conversation_memory WHERE session_id = ? ORDER BY 
        timestamp DESC LIMIT ?""",
        (session_id, limit)
    ).fetchall()
    return "\n".join(f"{role}: {msg}, {timestamp}" for role, message, timestamp in reversed(rows))

In [45]:
# Retrieve and keep track of user messages and assistant responses
def get_chat_history_tuples(session_id: str, limit: int = 6):
    """
    Return the most recent user/assistant message pairs for a session.

    The messages are returned in chronological order as:
    [
        (user_message, assistant_response),
        ...
    ]
    """

    rows = db.execute(
        """
        SELECT role, message, timestamp
        FROM (
            SELECT role, message, timestamp
            FROM conversation_memory
            WHERE session_id = ?
            ORDER BY timestamp DESC
            LIMIT ?
        )
        ORDER BY timestamp ASC
        """,
        (session_id, limit * 2)
    ).fetchall()

    history = []
    user_msg = None

    for role, message, timestamp in rows:
        if role == "user":
            user_msg = message

        elif role == "assistant" and user_msg is not None:
            history.append((user_msg, message))
            user_msg = None

    return history[-limit:]

In [53]:
def get_chat_history_text(session_id: str, limit: int = 6) -> str:
    """
    Return the most recent conversation messages as formatted text.
    """

    rows = db.execute(
        """
        SELECT role, message
        FROM (
            SELECT role, message, timestamp
            FROM conversation_memory
            WHERE session_id = ?
            ORDER BY timestamp DESC
            LIMIT ?
        )
        ORDER BY timestamp ASC
        """,
        (session_id, limit * 2)
    ).fetchall()

    return "\n".join(
        f"{role}: {message}"
        for role, message in rows
    )

#### Agents with Memory (database)

In [67]:
@tool
def policy_agent_tool(question: str, guest_type: str, loyalty: str, city: str, chat_history: list):
    """
    Tool for interpreting hotel policies based on guest information and location. It 
    retrieves and summarizes relevant policy information.
    """
    # Implementation for policy interpretation
    return policy_chain.invoke(
        {"question": question, 
         "chat_history": chat_history, 
         "guest_type": guest_type, 
         "loyalty": loyalty, 
         "city": city})["answer"]

In [66]:
@tool
def conversation_agent_tool(conversation_query: str, conversation_memory: str):
    """
    Tool for analyzing historical conversations to provide insights and guidance on how to handle 
    similar inquiries. It retrieves and summarizes relevant conversation information.
    """
    # Implementation for conversation analysis
    return conversation_agent.invoke(
        {"query": conversation_query, 
         "chat_history": conversation_memory})["result"]

#### Creating the Aggregator or Synthethizer Agent

In [35]:
aggregator_prompt = PromptTemplate(
    input_variables=["policy_agent_output", "conversation_agent_output","question"],
    
    template="""You are the FINAL RESPONSE AGGREGATOR AGENT for Aurora Hospitality. Your role is to combine
    policy interpretation and conversation analysis to provide a comprehensive response to user inquiries.
    You will receive outputs from the POLICY AGENT and the CONVERSATION AGENT, along with the original
    user question. Your task is to synthesize this information into a single, coherent response that
    addresses the user's query effectively and your final response must be in English.
    
    CORE PRINCIPLES

    Policy output defines what is allowed and not allowed, while conversation output provides insights
    into how similar situations were handled in the past.

    Conversation output provides real-world context, while policy output ensures adherence to official
    guidelines.
    
    If the conversation output contains information that is not aligned with the policy output, 
    prioritize the policy output in your final response but conversation can still guide tone used.

    When synthesizing the outputs, ensure that the final response is clear, concise, and actionable for the
    user.

    If policy is not found, refer the user to the customer support team for further assistance while you 
    answer the question you are confident of.

    If ambiguity arises between the two outputs or conversation not found, prioritize the policy output in your final response.

    When policy is unclear, provide a clear disclaimer in your final response, indicating that the information is based on
    the available policy documents and may be subject to change. Encourage the user to verify with official
    sources.

    INPUTS
    POLICY AGENT OUTPUT(authoritative rules): {policy_agent_output}
    CONVERSATION AGENT OUTPUT (how similar situations were handled in real customer-care scenarios): {conversation_agent_output}
    USER QUESTION: {question}

    TASK: 
    Synthesize the above information into a single, coherent response that addresses the user's query effectively.

    Sythesized a final response that:

    Is clear, concise, and actionable for the user

    Is factually accurate and adheres to the official policy guidelines

    Sounds professional, natural, and empathetic, reflecting the tone of Aurora Hospitality's customer service

    Reflects how real-world situations were handled in the past, providing context and guidance for the user
    
    Includes a clear disclaimer if the policy is unclear

    FINAL ANSWER REQUIREMENTS:

    Length: 3-5 sentences

    Tone: clear, natural and customer-friendly

    Do not mention the source of information in your final response or confidence scores.

    Do not mention the policy or conversation agent in your final response.

    If uncertainty exists refer the user to the customer support team for further assistance

    FINAL ANSWER:
    """
    )


In [36]:
from langchain.chains import LLMChain

aggregator_chain = LLMChain(
    llm=llm,
    prompt = aggregator_prompt
)

C:\Users\aghos\AppData\Local\Temp\ipykernel_23964\2114740537.py:3: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use :meth:`~RunnableSequence, e.g., `prompt | llm`` instead.
  aggregator_chain = LLMChain(


In [73]:
import asyncio


async def run_agents_parallel(
    question: str,
    guest_type: str,
    loyalty: str,
    city: str,
    chat_history_tuples,
    chat_history_text: str
):
    """
    Run the policy agent and conversation agent concurrently.

    The policy tool expects the key 'question'.
    The RetrievalQA conversation agent expects the key 'query'.
    """

    # Input expected by policy_agent_tool
    policy_input = {
        "question": question,
        "guest_type": guest_type,
        "loyalty": loyalty,
        "city": city,
        "chat_history":chat_history_tuples
    }

    # Include conversation memory in the query sent to RetrievalQA
    conversation_query = f"""
Guest type: {guest_type}
Loyalty tier: {loyalty}
City: {city}
Recent conversation history:
{chat_history_text or "No previous conversation history."}

Guest question:
{question}
""".strip()

    # Input expected by RetrievalQA
    conversation_input = {
        "query": conversation_query
    }

    policy_task = asyncio.to_thread(
        policy_agent_tool.invoke,
        policy_input
    )

    conversation_task = asyncio.to_thread(
        conversation_agent.invoke,
        conversation_input
    )

    policy_result, conversation_result = await asyncio.gather(
        policy_task,
        conversation_task
    )

    # Normalise policy agent output
    if isinstance(policy_result, dict):
        policy_output = (
            policy_result.get("text")
            or policy_result.get("answer")
            or policy_result.get("result")
            or policy_result
        )
    else:
        policy_output = str(policy_result)

    # RetrievalQA normally returns {"query": ..., "result": ...}
    if isinstance(conversation_result, dict):
        conversation_output = (
            conversation_result.get("result")
            or conversation_result.get("answer")
            or conversation_result.get("text")
            or conversation_result
        )
    else:
        conversation_output = str(conversation_result)

    return policy_output, conversation_output

In [74]:
async def agentic_rag_answer(
    question: str,
    guest_type: str,
    loyalty: str,
    city: str,
    session_id: str
):
    chat_history_tuples = get_chat_history_tuples(
        session_id=session_id,
        limit=6
    )

    chat_history_text = get_chat_history_text(
        session_id=session_id,
        limit=6
    )

    policy_output, conversation_output = await run_agents_parallel(
        question=question,
        guest_type=guest_type,
        loyalty=loyalty,
        city=city,
        chat_history_tuples=chat_history_tuples,
        chat_history_text=chat_history_text
    )

    final_answer = aggregator_chain.invoke(
        {
            "policy_agent_output": policy_output,
            "conversation_agent_output": conversation_output,
            "conversation_memory": chat_history_text,
            "question": question
        }
    )

    if isinstance(final_answer, dict):
        answer_text = (
            final_answer.get("text")
            or final_answer.get("answer")
            or final_answer.get("result")
        )
    else:
        answer_text = str(final_answer)

    if not answer_text:
        raise ValueError(
            f"No recognised answer field returned by aggregator_chain: "
            f"{final_answer}"
        )

    store_conversation(session_id, "user", question)
    store_conversation(session_id, "assistant", answer_text)

    return {
        "text": answer_text,
        "policy_output": policy_output,
        "conversation_output": conversation_output
    }

In [75]:
session_id = "guest_001"

response = await agentic_rag_answer(
    question="Amenities & services (wifi, breakfast, gym, pool, spa)",
    guest_type="Business",
    loyalty="Gold",
    city="Sydney",
    session_id=session_id
)

print(response["text"])


Thank you for your inquiry about our amenities and services. We offer a pool area with poolside service available from 11:00 AM to 6:00 PM, and spa services daily from 8:00 AM to 8:00 PM, with advance reservations recommended. While we do have fitness centers, specific details about gym facilities are not provided. For information regarding Wi-Fi availability and breakfast services, please reach out to our customer support team, as these details can vary based on your rate and package. If you have any further questions, feel free to ask!


In [76]:
session_id = "guest_001"

response = await agentic_rag_answer(
    question="Anyways, what time is breakfast going to be served",
    guest_type="Business",
    loyalty="Gold",
    city="Sydney",
    session_id=session_id
)

print(response["text"])

Breakfast is served at our hotel during the following times: on weekdays from 6:30 AM to 10:30 AM, and on weekends from 7:00 AM to 11:00 AM. If you have any specific questions about your stay or if there are any seasonal adjustments, please let me know, and I’ll be happy to assist you further. Enjoy your stay!


In [77]:
session_id = "guest_001"

response = await agentic_rag_answer(
    question="What about Lunch?",
    guest_type="Business",
    loyalty="Gold",
    city="Sydney",
    session_id=session_id
)

print(response["text"])

Lunch is served in the Main Restaurant from 11:30 AM to 2:30 PM. If you have any specific dietary preferences or would like more information about the menu, please feel free to ask! We're here to help, so don't hesitate to reach out if you have any further questions.


In [78]:
session_id = "guest_001"

response = await agentic_rag_answer(
    question="And dinner?",
    guest_type="Business",
    loyalty="Gold",
    city="Sydney",
    session_id=session_id
)

print(response["text"])

Dinner is served in the Main Restaurant from 5:30 PM to 10:00 PM. If you have any specific preferences or would like more information about the menu, please feel free to ask! We're here to help and ensure you have a wonderful dining experience.


The primary difference is that ConversationalRetrievalChain dynamically rewrites the questions using chat history before searching the database, whereas RetrievalQA searches the database using the exact, raw query. RetrievalQA handles isolated, single-turn questions. ConversationalRetrievalChain handles ongoing, multi-turn conversations.

Behind the Scenes: 

RetrievalQA: RetrievalQA runs a straightforward, single-step retrieval pipeline. It assumes every question stands completely on its own.

[ User Query ] ──► [ Vector Store Search ] ──► [ LLM + Context ] ──► [ Final Answer ]

The Vector Search: When a query is passed (e.g., "How much did we spend on it?"). RetrievalQA converts this exact string into an embedding vector. It immediately queries the vector database.

The Retrieval Flaw: If the query contains pronouns like "it," "they," or "he," the vector database performs poorly. It searches for the literal word "it" instead of the actual subject discussed in previous turns.

The Generation: The system passes whatever chunks it finds to the LLM alongside the raw question. The LLM generates the final response.

ConversationalRetrievalChain:

ConversationalRetrievalChain runs a two-step LLM process. It uses a specialized sub-chain to resolve pronouns and context before searching the database.                        
                        ┌──────────────────┐
                        │   Chat History   │
                        └────────┬─────────┘
                                 ▼
[ New User Query ] ──► [ LLM: Question Re-writer ] ──► [ Standalone Query ]
                                                               │
                                                               ▼
[ Final Answer ] ◄── [ LLM + Context ] ◄── [ Vector Store ] ◄──┘

Step 1: The Context condensation (The "Secret Sauce")When a new question is submitted, the chain does not search the database right away. Instead, it bundles two things together and sends them to the LLM:
- The entire Chat History (past questions and answers)
- Your New Follow-Up Question

It instructs the LLM: "Read this history and rewrite the new question so it can be understood completely on its own, without needing the history.

"Example: If the history is talking about "Project Alpha", and the new question is "How much did we spend on it?", the re-writer LLM outputs a brand-new string: "How much money was spent on Project Alpha?"

Step 2: The Standard RAG Pipeline 

The chain takes that newly generated, standalone question and passes it into a standard retrieval flow:
- It embeds the rewritten question and searches the vector database.
- The database returns highly accurate, relevant text chunks because the vague pronouns were resolved.
- The chain sends these precise chunks, along with the original question and chat history, to the final LLM to generate a natural, context-aware reply.

Summary of LLM Calls

RetrievalQA: Uses exactly one LLM call per user interaction (to answer the question based on the retrieved documents).

ConversationalRetrievalChain: Uses two LLM calls for every follow-up interaction. Call one rewrites the question; call two generates the final answer.